In [3]:
# pip install redis

In [43]:
import redis
import json

In [13]:
redis_client = redis.Redis(host="localhost", port=6379, decode_responses=True)

In [14]:
redis_client.ping()

True

In [17]:
STREAM_KEY = 'cdc.public.mapping_id'
last_id = '0-0'

In [18]:
messages = redis_client.xread({STREAM_KEY: last_id}, count=10, block=5000)

In [19]:
messages

[['cdc.public.mapping_id',
  [('1781925133716-0',
    {'default': '{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int64","optional":true,"field":"index"},{"type":"int64","optional":true,"field":"id"},{"type":"string","optional":true,"field":"uuid"}],"optional":true,"name":"cdc.public.mapping_id.Value","field":"before"},{"type":"struct","fields":[{"type":"int64","optional":true,"field":"index"},{"type":"int64","optional":true,"field":"id"},{"type":"string","optional":true,"field":"uuid"}],"optional":true,"name":"cdc.public.mapping_id.Value","field":"after"},{"type":"struct","fields":[{"type":"string","optional":false,"field":"version"},{"type":"string","optional":false,"field":"connector"},{"type":"string","optional":false,"field":"name"},{"type":"int64","optional":false,"field":"ts_ms"},{"type":"string","optional":true,"name":"io.debezium.data.Enum","version":1,"parameters":{"allowed":"true,last,false,incremental"},"default":"false","field":"snapshot"},{"type"

## message
- [0] for list in list 
- [0][0] key name
- [0][1] data

In [37]:
data = messages[0][1]
data

[('1781925133716-0',
  {'default': '{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int64","optional":true,"field":"index"},{"type":"int64","optional":true,"field":"id"},{"type":"string","optional":true,"field":"uuid"}],"optional":true,"name":"cdc.public.mapping_id.Value","field":"before"},{"type":"struct","fields":[{"type":"int64","optional":true,"field":"index"},{"type":"int64","optional":true,"field":"id"},{"type":"string","optional":true,"field":"uuid"}],"optional":true,"name":"cdc.public.mapping_id.Value","field":"after"},{"type":"struct","fields":[{"type":"string","optional":false,"field":"version"},{"type":"string","optional":false,"field":"connector"},{"type":"string","optional":false,"field":"name"},{"type":"int64","optional":false,"field":"ts_ms"},{"type":"string","optional":true,"name":"io.debezium.data.Enum","version":1,"parameters":{"allowed":"true,last,false,incremental"},"default":"false","field":"snapshot"},{"type":"string","optional":false,"fie

# data => list 
- data[0] 1st element
- data[0][0] -> id
- data[0][1] -> data(dict)

In [55]:
elm = json.loads(data[0][1]['default'])
elm

{'schema': {'type': 'struct',
  'fields': [{'type': 'struct',
    'fields': [{'type': 'int64', 'optional': True, 'field': 'index'},
     {'type': 'int64', 'optional': True, 'field': 'id'},
     {'type': 'string', 'optional': True, 'field': 'uuid'}],
    'optional': True,
    'name': 'cdc.public.mapping_id.Value',
    'field': 'before'},
   {'type': 'struct',
    'fields': [{'type': 'int64', 'optional': True, 'field': 'index'},
     {'type': 'int64', 'optional': True, 'field': 'id'},
     {'type': 'string', 'optional': True, 'field': 'uuid'}],
    'optional': True,
    'name': 'cdc.public.mapping_id.Value',
    'field': 'after'},
   {'type': 'struct',
    'fields': [{'type': 'string', 'optional': False, 'field': 'version'},
     {'type': 'string', 'optional': False, 'field': 'connector'},
     {'type': 'string', 'optional': False, 'field': 'name'},
     {'type': 'int64', 'optional': False, 'field': 'ts_ms'},
     {'type': 'string',
      'optional': True,
      'name': 'io.debezium.data

In [59]:
elm['payload']['after']

{'index': 0, 'id': 0, 'uuid': '5039f86c-379a-462a-a8c5-a7195f9ed43e'}

In [84]:
import pandas as pd
import s3fs

In [85]:
# pip install s3fs

In [70]:
df = pd.DataFrame([
    json.loads(elm[1]['default'])['payload']['after']
    for elm in data
])
df

,index,id,uuid
0,0,0,5039f86c-379a-462a-a8c5-a7195f9ed43e
1,1,1,e58a4d31-3dde-4435-8eb9-375642eb238f
2,2,2,2fb18656-b37e-44ea-be63-daedb0248def
3,3,3,d5a9d0da-ce82-447a-b764-b3ad028fc300
4,4,4,a458cc4c-b2cb-4a81-a9c0-49eb79f03fa5
5,5,5,f56762ca-60b5-4486-8f21-2ead00932a3a
6,6,6,e1ce5f1c-f725-4fee-b528-bdbc1326aaa4
7,7,7,35395f82-ada9-49f7-b261-5e95c8725996
8,8,8,ed220d84-cbb8-4354-ae4b-04d897772ae0
9,9,9,607f8935-0553-4dc9-910a-3e42db5377b6


In [77]:
datas = []
for event_id, event_data in messages[0][1]:
    data = json.loads(event_data['default'])
    datas.append(data)
    redis_client.xdel(STREAM_KEY, event_id)

In [80]:
df = pd.DataFrame([
    elm.get('payload').get('after')
for elm in datas])
df

,index,id,uuid
0,0,0,5039f86c-379a-462a-a8c5-a7195f9ed43e
1,1,1,e58a4d31-3dde-4435-8eb9-375642eb238f
2,2,2,2fb18656-b37e-44ea-be63-daedb0248def
3,3,3,d5a9d0da-ce82-447a-b764-b3ad028fc300
4,4,4,a458cc4c-b2cb-4a81-a9c0-49eb79f03fa5
5,5,5,f56762ca-60b5-4486-8f21-2ead00932a3a
6,6,6,e1ce5f1c-f725-4fee-b528-bdbc1326aaa4
7,7,7,35395f82-ada9-49f7-b261-5e95c8725996
8,8,8,ed220d84-cbb8-4354-ae4b-04d897772ae0
9,9,9,607f8935-0553-4dc9-910a-3e42db5377b6


In [86]:
STREAM_KEY_ORACLE = 'cdc-oracle.C__APPUSER.CUSTOMERS'

In [108]:
messages = redis_client.xread({STREAM_KEY_ORACLE: last_id}, count=10, block=5000)
messages

[['cdc-oracle.C__APPUSER.CUSTOMERS',
  [('1781960944951-0',
    {'{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int32","optional":false,"field":"scale"},{"type":"bytes","optional":false,"field":"value"}],"optional":false,"name":"io.debezium.data.VariableScaleDecimal","version":1,"doc":"Variable scaled decimal","field":"ID"}],"optional":false,"name":"cdc-oracle.C__APPUSER.CUSTOMERS.Key"},"payload":{"ID":{"scale":0,"value":"AQ=="}}}': '{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int32","optional":false,"field":"scale"},{"type":"bytes","optional":false,"field":"value"}],"optional":false,"name":"io.debezium.data.VariableScaleDecimal","version":1,"doc":"Variable scaled decimal","field":"ID"},{"type":"string","optional":true,"field":"NAME"},{"type":"string","optional":true,"field":"EMAIL"}],"optional":true,"name":"cdc-oracle.C__APPUSER.CUSTOMERS.Value","field":"before"},{"type":"struct","fields":[{"type":"stru

In [131]:
datas = []
for event_id, event_data in messages[0][1]:
    data = json.loads(event_data[list(event_data.keys())[0]])
    datas.append(data)
    redis_client.xdel(STREAM_KEY_ORACLE, event_id)

In [132]:
df = pd.DataFrame([
    elm.get('payload').get('after')
for elm in datas])
df

,ID,NAME,EMAIL
0,"{'scale': 0, 'value': 'AQ=='}",Test User,test@example.com
1,"{'scale': 0, 'value': 'FQ=='}",Robin Herrera,kenneth37@example.org
2,"{'scale': 0, 'value': 'Fg=='}",Kimberly Cameron,qmccormick@example.org
3,"{'scale': 0, 'value': 'Fw=='}",Janice Lam,colleen23@example.org
4,"{'scale': 0, 'value': 'GA=='}",Shelia Martinez,jameshenderson@example.org
5,"{'scale': 0, 'value': 'GQ=='}",Cassidy Jenkins,cmarks@example.net
6,"{'scale': 0, 'value': 'Gg=='}",Robin Obrien,sarahhamilton@example.net
7,"{'scale': 0, 'value': 'Gw=='}",Richard Moore,wmartinez@example.org
8,"{'scale': 0, 'value': 'HA=='}",Aaron Jordan,hudsonzoe@example.net
9,"{'scale': 0, 'value': 'HQ=='}",Shawn Mcgee,psanchez@example.com
